
# GE Pruebas Rapidas (XLSX/Parquet + Regla)

Flujo minimo:
1. Cargar un archivo XLSX o Parquet
2. Traer una regla desde `regla_negocio_great_expectation` por `Pkregla`
3. Ejecutar validacion GE
4. Mostrar `total`, `cumple`, `no_cumple`, `no_aplica`, `pct_calidad`


In [0]:
%pip install great_expectations==1.6.3 openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/57.7 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/4.9 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━ 4.3/4.9 MB 128.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 4.9/4.9 MB 126.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/250.9 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/813.6 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.6/813.6 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/134.9 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/90.6 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━

In [0]:
%restart_python

In [0]:
import json
from datetime import datetime

import pandas as pd
import great_expectations as gx
import great_expectations as ge
from pyspark.sql.functions import col, coalesce, lit, trim, expr
from great_expectations.core.run_identifier import RunIdentifier
from great_expectations.exceptions import DataContextError



In [0]:
# Widgets
# source_path: path del archivo/fuente (dbfs:/..., /dbfs/..., abfss://...)
# source_format: auto | xlsx | parquet
# source_xlsx: (compatibilidad) path del archivo xlsx
# sheet_name: indice o nombre de hoja (solo aplica a xlsx)
# rules_table: esquema.tabla de reglas de gobierno
# pkregla: id de regla (ej: 217)

dbutils.widgets.text("source_path", "")
dbutils.widgets.text("source_format", "auto")
dbutils.widgets.text("source_xlsx", "")
dbutils.widgets.text("sheet_name", "0")
dbutils.widgets.text("rules_table", "dev_arqanalitica.gobiernodato.regla_negocio_great_expectation")
dbutils.widgets.text("pkregla", "217")

SOURCE_PATH = dbutils.widgets.get("source_path").strip() or dbutils.widgets.get("source_xlsx").strip()
SOURCE_FORMAT = dbutils.widgets.get("source_format").strip().lower() or "auto"
SHEET_NAME = dbutils.widgets.get("sheet_name").strip()
RULES_TABLE = dbutils.widgets.get("rules_table").strip()
PKREGLA = dbutils.widgets.get("pkregla").strip()

assert SOURCE_PATH, "Debes informar source_path (o source_xlsx por compatibilidad)"
assert PKREGLA, "Debes informar pkregla"


In [0]:
def load_rule_from_table(rules_table: str, pkregla: str) -> dict:
    q = f"SELECT * FROM {rules_table} WHERE Pkregla = {pkregla} LIMIT 1"
    row = spark.sql(q).first()
    if row is None:
        raise ValueError(f"No se encontro regla con Pkregla={pkregla} en {rules_table}")
    return row.asDict(recursive=True)


def _sql_quote(v):
    return str(v).replace("'", "''")


def _ref_values(dominio: str, codigo: str):
    df_ref = spark.sql(f"""
        SELECT DISTINCT valor
        FROM dev_arqanalitica.gobiernodato.lista_referencia
        WHERE dominio = '{_sql_quote(dominio)}' AND codigo = '{_sql_quote(codigo)}'
    """)
    return [r.valor for r in df_ref.collect() if r.valor is not None]


def _ref_desc_val(dominio: str, codigo: str):
    df_ref = spark.sql(f"""
        SELECT DISTINCT valor, descripcion
        FROM dev_arqanalitica.gobiernodato.lista_referencia
        WHERE dominio = '{_sql_quote(dominio)}' AND codigo = '{_sql_quote(codigo)}'
    """)
    return [(r.valor, r.descripcion) for r in df_ref.collect() if r.valor is not None]


def _expectativa_valor_unico(columna, **_):
    return ge.expectations.ExpectColumnValuesToBeUnique(column=columna)


def _expectativa_valores_por_referencia(columna, dominio, codigo, **_):
    return ge.expectations.ExpectColumnValuesToBeInSet(column=columna, value_set=_ref_values(dominio, codigo))


def _expectativa_valores_por_referencia_caracteres(columna, referencia, caracteres, **_):
    if isinstance(referencia, list):
        vals = [str(v) for v in referencia if v is not None]
    else:
        vals = [x.strip() for x in str(referencia).split(",") if x.strip()]
    pref = {v[:int(caracteres)] for v in vals if v}
    regex = f"^({'|'.join(pref)})" if pref else r"^$"
    return ge.expectations.ExpectColumnValuesToMatchRegex(column=columna, regex=regex)


def _expectativa_valores_con_patron(columna, patron, **_):
    return ge.expectations.ExpectColumnValuesToMatchRegex(column=columna, regex=patron)


def _expectativa_valores_tabla_equivalencia(columnas, dominio, codigo, separador="|", condicion=None, **_):
    ref_vals = _ref_values(dominio, codigo)
    col_expr = f"CONCAT_WS('{_sql_quote(separador)}', {', '.join(columnas)})"
    in_list = ",".join([f"'{_sql_quote(v)}'" for v in ref_vals]) if ref_vals else "''"
    where_base = f"{col_expr} NOT IN ({in_list})"
    where_clause = f"{condicion} AND {where_base}" if condicion else where_base
    sql = f"SELECT * FROM {{batch}} WHERE {where_clause}"
    return ge.expectations.UnexpectedRowsExpectation(unexpected_rows_query=sql)


def _expectativa_valores_no_nulos(columna, **_):
    return ge.expectations.ExpectColumnValuesToNotBeNull(column=columna)


def _expectativa_valores_tipo(columna, tipo, **_):
    return ge.expectations.ExpectColumnValuesToBeOfType(column=columna, type_=tipo)


def _expectativa_valores_longitud_entre(columna, min_valor, max_valor, **_):
    return ge.expectations.ExpectColumnValueLengthsToBeBetween(column=columna, min_value=min_valor, max_value=max_valor)


def _expectativa_valores_longitud_igual(columna, longitud, **_):
    return ge.expectations.ExpectColumnValueLengthsToEqual(column=columna, value=longitud)


def _expectativa_columna_valores_entre(columna, min_valor, max_valor, **_):
    return ge.expectations.ExpectColumnValuesToBeBetween(column=columna, min_value=min_valor, max_value=max_valor)


def _expectativa_columnas_esperadas(columnas, **_):
    return ge.expectations.ExpectTableColumnsToMatchSet(column_set=columnas)


def _expectativa_combinacion_columnas_unicas(columnas, **_):
    return ge.expectations.ExpectCompoundColumnsToBeUnique(column_list=columnas)


def _expectativa_valores_distintos_por_referencia(columna, dominio, codigo, **_):
    return ge.expectations.ExpectColumnValuesToNotBeInSet(column=columna, value_set=_ref_values(dominio, codigo))


def _expectativa_cantidad_filas_entre(min_valor, max_valor, **_):
    return ge.expectations.ExpectTableRowCountToBeBetween(min_value=min_valor, max_value=max_valor)


def _expectativa_valores_no_vacios(columna, **_):
    return ge.expectations.ExpectColumnValuesToNotMatchRegex(column=columna, regex=r"^\s*$")


def _expectativa_con_condicional_y_patron(condicion, columna, patron, **_):
    return ge.expectations.ExpectColumnValuesToMatchRegex(
        column=columna, regex=patron, row_condition=condicion, condition_parser="spark"
    )


def _expectativa_condicional_tabla_referencia(condicion, columna, dominio, codigo, **_):
    return ge.expectations.ExpectColumnValuesToBeInSet(
        column=columna,
        value_set=_ref_values(dominio, codigo),
        row_condition=condicion,
        condition_parser="spark",
    )


def _expectativa_condicional_valores_en_lista(condicion, columna, lista, **_):
    return ge.expectations.ExpectColumnValuesToBeInSet(
        column=columna,
        value_set=lista,
        row_condition=condicion,
        condition_parser="spark",
    )


def _expectativa_primera_palabra_en_referencia_con_condicion(condicion, columna, dominio, referencia, **_):
    vals = _ref_values(dominio, referencia)
    palabras = list({str(v).split()[0] for v in vals if str(v).strip()})
    regex = f"^({'|'.join(palabras)})" if palabras else r"^$"
    return ge.expectations.ExpectColumnValuesToMatchRegex(
        column=columna, regex=regex, row_condition=condicion, condition_parser="spark"
    )


def _expectativa_valores_no_permitidos(columna, valores, condicion=None, **_):
    return ge.expectations.ExpectColumnValuesToNotBeInSet(
        column=columna, value_set=valores, row_condition=condicion, condition_parser="spark"
    )


def _expectativa_apellido_nombre_condicional(NAME1, NAME3, NAME4, condicion_sql=None, **_):
    if condicion_sql:
        where_clause = f"WHERE {condicion_sql} AND {NAME1} != CONCAT(REPLACE({NAME4}, ',', ' '), ' ', REPLACE({NAME3}, ',', ' '))"
    else:
        where_clause = f"WHERE {NAME1} != CONCAT(REPLACE({NAME4}, ',', ' '), ' ', REPLACE({NAME3}, ',', ' '))"
    sql = f"SELECT * FROM {{batch}} {where_clause}"
    return ge.expectations.UnexpectedRowsExpectation(unexpected_rows_query=sql)


def _expectativa_valores_nulos_con_condicion(condicion, columna, **_):
    return ge.expectations.ExpectColumnValuesToBeNull(
        column=columna, row_condition=condicion, condition_parser="spark"
    )


def _expectativa_valores_por_referencia_caracteres_con_condicion(condicion, columna, dominio, codigo, caracteres, referencia, **_):
    pairs = _ref_desc_val(dominio, codigo)
    descs = [d for _, d in pairs if d is not None]
    prefijos = list({str(v)[:int(caracteres)] for v, _ in pairs if v is not None})
    regex = f"^({'|'.join(prefijos)})" if prefijos else r"^$"
    in_desc = ",".join([f"'{_sql_quote(d)}'" for d in descs]) if descs else "''"
    row_condition = f"{condicion} AND {referencia} IN ({in_desc})"
    return ge.expectations.ExpectColumnValuesToMatchRegex(
        column=columna, regex=regex, row_condition=row_condition, condition_parser="spark"
    )


FUNCIONES_EXPECTATIVAS = {
    "expectativa_valores_con_patron": _expectativa_valores_con_patron,
    "expectativa_valor_unico": _expectativa_valor_unico,
    "expectativa_valores_por_referencia_caracteres_con_condicion": _expectativa_valores_por_referencia_caracteres_con_condicion,
    "expectativa_valores_por_referencia": _expectativa_valores_por_referencia,
    "expectativa_valores_no_nulos": _expectativa_valores_no_nulos,
    "expectativa_con_condicional_y_patron": _expectativa_con_condicional_y_patron,
    "expectativa_valores_no_permitidos": _expectativa_valores_no_permitidos,
    "expectativa_primera_palabra_en_referencia_con_condicion": _expectativa_primera_palabra_en_referencia_con_condicion,
    "expectativa_valores_nulos_con_condicion": _expectativa_valores_nulos_con_condicion,
    "expectativa_condicional_tabla_referencia": _expectativa_condicional_tabla_referencia,
    "expectativa_condicional_valores_en_lista": _expectativa_condicional_valores_en_lista,
    "expectativa_valores_tabla_equivalencia": _expectativa_valores_tabla_equivalencia,
    "expectativa_apellido_nombre_condicional": _expectativa_apellido_nombre_condicional,
    "expectativa_valores_por_referencia_caracteres": _expectativa_valores_por_referencia_caracteres,
    "expectativa_valores_tipo": _expectativa_valores_tipo,
    "expectativa_valores_longitud_entre": _expectativa_valores_longitud_entre,
    "expectativa_valores_longitud_igual": _expectativa_valores_longitud_igual,
    "expectativa_columna_valores_entre": _expectativa_columna_valores_entre,
    "expectativa_columnas_esperadas": _expectativa_columnas_esperadas,
    "expectativa_combinacion_columnas_unicas": _expectativa_combinacion_columnas_unicas,
    "expectativa_valores_distintos_por_referencia": _expectativa_valores_distintos_por_referencia,
    "expectativa_cantidad_filas_entre": _expectativa_cantidad_filas_entre,
    "expectativa_valores_no_vacios": _expectativa_valores_no_vacios,
}


def build_expectation_from_governance_rule(rule_row: dict):
    tipo = (rule_row.get("NombreReglaGreatExpectation")
            or rule_row.get("NOMBREREGLAGREATEXPECTATION")
            or rule_row.get("tipo_expectativa")
            or rule_row.get("TIPO_EXPECTATIVA")
            or "")
    tipo = str(tipo).strip().lower()

    if not tipo:
        raise ValueError(f"No encontre tipo de expectativa. Keys={list(rule_row.keys())}")

    raw_params = rule_row.get("parametros") or rule_row.get("PARAMETROS") or "{}"
    params = json.loads(raw_params) if isinstance(raw_params, str) else raw_params

    func = FUNCIONES_EXPECTATIVAS.get(tipo)
    if not func:
        soportadas = ", ".join(sorted(FUNCIONES_EXPECTATIVAS.keys()))
        raise ValueError(f"tipo_expectativa no soportado: {tipo}. Soportadas: {soportadas}")

    return func(**params), params, tipo


In [0]:
DATASOURCE_NAME = "spark_datasource_sandbox"
BATCH_DEFINITION_NAME = "full_table_batch"


def get_test_context():
    # Contexto efimero para pruebas, sin generar Data Docs
    try:
        return gx.get_context(mode="ephemeral")
    except TypeError:
        return gx.get_context()


def get_or_create_datasource(context, name=DATASOURCE_NAME):
    try:
        return context.data_sources.get(name)
    except KeyError:
        return context.data_sources.add_spark(name=name)


def get_or_create_asset(datasource, asset_name):
    if asset_name not in [a.name for a in datasource.assets]:
        return datasource.add_dataframe_asset(name=asset_name)
    return datasource.get_asset(asset_name)


def get_or_create_batch_definition(data_asset, batch_definition_name=BATCH_DEFINITION_NAME):
    if batch_definition_name not in [bd.name for bd in data_asset.batch_definitions]:
        return data_asset.add_batch_definition_whole_dataframe(batch_definition_name)
    return data_asset.get_batch_definition(batch_definition_name)


def create_or_update_suite(context, suite_name, expectations):
    try:
        suite = context.suites.get(suite_name)
        suite.expectations = []
    except DataContextError:
        suite = gx.ExpectationSuite(name=suite_name)

    for exp in expectations:
        suite.add_expectation(exp)

    context.suites.add_or_update(suite)
    return suite


def create_validation_definition(context, batch_definition, suite, validation_name):
    try:
        return context.validation_definitions.get(validation_name)
    except DataContextError:
        validation_def = gx.ValidationDefinition(
            data=batch_definition,
            suite=suite,
            name=validation_name,
        )
        context.validation_definitions.add(validation_def)
        return validation_def


def create_or_update_checkpoint(context, checkpoint_name, validation_definitions):
    try:
        checkpoint = context.checkpoints.get(checkpoint_name)
        checkpoint.validation_definitions = validation_definitions
        checkpoint.actions = []
        checkpoint.save()
    except DataContextError:
        checkpoint = gx.Checkpoint(
            name=checkpoint_name,
            validation_definitions=validation_definitions,
            actions=[],
            result_format={"result_format": "COMPLETE"},
        )
        context.checkpoints.add(checkpoint)
    return checkpoint


def generate_run_identifier(prefix="sandbox"):
    return RunIdentifier(run_name=f"{prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}")


In [0]:
def quality_counts_for_conditional_regex(df_in, condicion: str, columna: str, patron: str):
    total = df_in.count()
    aplica = df_in.where(expr(condicion))
    cumple = aplica.where(trim(coalesce(col(columna).cast("string"), lit(""))).rlike(patron)).count()
    no_cumple = aplica.where(~trim(coalesce(col(columna).cast("string"), lit(""))).rlike(patron)).count()
    no_aplica = total - (cumple + no_cumple)
    pct = round((cumple / (cumple + no_cumple) * 100), 1) if (cumple + no_cumple) > 0 else 0.0

    return spark.createDataFrame(
        [(total, cumple, no_cumple, no_aplica, pct)],
        ["total", "cumple", "no_cumple", "no_aplica", "pct_calidad"],
    )


In [0]:
import pandas as pd

def _dbfs_to_local(path: str) -> str:
    if path.startswith("dbfs:/"):
        return "/dbfs/" + path[len("dbfs:/"):]
    return path

def load_input_to_spark(path: str, source_format: str = "auto", sheet_name: str = "0"):
    fmt = (source_format or "auto").strip().lower()
    if fmt == "auto":
        p = path.lower()
        fmt = "parquet" if ".parquet" in p else "xlsx"

    if fmt == "parquet":
        return spark.read.parquet(path)

    if fmt == "xlsx":
        local_path = _dbfs_to_local(path)
        sheet = int(sheet_name) if str(sheet_name).isdigit() else sheet_name
        pdf = pd.read_excel(local_path, sheet_name=sheet, engine="openpyxl")
        return spark.createDataFrame(pdf)

    raise ValueError(f"Formato no soportado: {fmt}. Usa auto, xlsx o parquet")


In [0]:
print("load_input_to_spark" in globals())


True


In [0]:
# 1) Carga fuente (xlsx/parquet)

df = load_input_to_spark(SOURCE_PATH, SOURCE_FORMAT, SHEET_NAME)
print(f"Filas: {df.count()}, Columnas: {len(df.columns)}")
display(df.limit(20))


/databricks/spark/python/pyspark/sql/pandas/utils.py:51: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):


Filas: 99, Columnas: 139


LIFNR,LAND1,NAME1,NAME2,NAME3,NAME4,ORT01,ORT02,PFACH,PSTL2,PSTLZ,REGIO,SORTL,STRAS,ADRNR,MCOD1,MCOD2,MCOD3,ANRED,BAHNS,BBBNR,BBSNR,BEGRU,BRSCH,BUBKZ,DATLT,DTAMS,DTAWS,ERDAT,ERNAM,ESRNR,KONZS,KTOKK,KUNNR,LNRZA,LOEVM,SPERR,SPERM,SPRAS,STCD1,STCD2,STKZA,STKZU,TELBX,TELF1,TELF2,TELFX,TELTX,TELX1,XCPDK,XZEMP,VBUND,FISKN,STCEG,STKZN,SPERQ,GBORT,GBDAT,SEXKZ,KRAUS,REVDB,QSSYS,KTOCK,PFORT,WERKS,LTSNA,WERKR,PLKAL,DUEFL,TXJCD,SPERZ,SCACD,SFRGR,LZONE,XLFZA,DLGRP,FITYP,STCDT,REGSS,ACTSS,STCD3,STCD4,STCD5,IPISP,TAXBS,PROFS,STGDL,EMNFR,LFURL,J_1KFREPRE,J_1KFTBUS,J_1KFTIND,CONFS,UPDAT,UPTIM,NODEL,QSSYSDAT,PODKZB,FISKU,STENR,CARRIER_CONF,MIN_COMP,TERM_LI,CRC_NUM,CVP_XBLCK,RG,EXP,UF,RGDATE,RIC,RNE,RNEDATE,CNAE,LEGALNAT,CRTN,ICMSTAXPAY,INDTYP,TDT,COMSIZE,DECREGPC,J_SC_CAPITAL,J_SC_CURRENCY,ALC,PMT_OFFICE,PPA_RELEVANT,PSOFG,PSOIS,PSON1,PSON2,PSON3,PSOVN,PSOTL,PSOHS,PSOST,TRANSPORT_CHAIN,STAGING_TIME,SCHEDULING_TYPE,SUBMI_RELEVANT,OPFLAG
210000,CO,SUMINISTROS DE COLOMBIA S.A.S,null,null,null,SABANETA,null,null,null,5631,05,890900120,CR 49 67 SUR 520 AV REGIONAL,27097,SUMINISTROS DE COLOMBIA S,null,SABANETA,Señor (es),null,0,0,null,K002,0,null,null,null,2014-12-27T00:00:00Z,LGIL,null,null,Z007,210000.0,null,null,null,null,ES,8909001207,null,null,null,null,(4)3058200,3138480130,9999999999,null,null,null,null,SM00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,X,null,null,null,null,null,null,1,PJ,N,null,null,null,null,null,null,0,null,Z1,null,null,null,null,null,null,null,00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,null,null,0,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,null,null
210004,CO,LOCERIA COLOMBIANA S.A.S,null,null,null,MEDELLIN,null,MDE,null,5129,05,890900085,CR 54 129 SUR 51,27101,LOCERIA COLOMBIANA S.A.S,null,MEDELLIN,Señor (es),null,0,0,null,K002,0,null,null,null,2014-12-27T00:00:00Z,LGIL,null,null,Z007,210004.0,null,null,null,null,ES,8909000857,null,null,null,null,(4)3787800,0,(4)2781558,null,null,null,null,LC00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,X,null,null,null,null,null,null,1,PJ,N,null,null,null,null,null,null,0,null,Z1,null,null,null,85-MX-Otros,null,null,null,00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,null,null,0,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,null,null
210005,MX,CERAMICAS Y MATERIALES CORONA S.A.,null,null,null,CIUDAD DE MEXICO,null,null,null,3810,DF,CMC991112C,MONTECITO 38 PISO 38 OFICINAS 31 Y,59593,CERAMICAS Y MATERIALES CO,null,CIUDAD DE MEXICO,Señor (es),null,0,0,null,K002,0,null,null,null,2015-01-05T00:00:00Z,LGIL,null,null,Z007,210005.0,null,null,null,null,ES,CMC991112CF2,null,null,null,null,52590001219,999999999999,525590001218,null,null,null,null,CMC0,null,null,X,null,null,null,null,null,null,null,null,null,null,null,null,null,X,null,null,null,null,null,null,7,PJ,N,null,null,null,null,null,null,0,null,Z1,null,null,null,85-MX-Otros,null,null,null,00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,null,null,0,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,null,null
210007,CO,CORONA INDUSTRIAL S.A.S,null,null,null,BOGOTA D.C.,null,null,null,11001,11,900696296,CL 100 8 A 55 TO C P 9,27102,CORONA INDUSTRIAL S.A.S,null,BOGOTA D.C.,Señor (es),null,0,0,null,K002,0,null,null,null,2014-12-27T00:00:00Z,LGIL,null,null,Z007,210007.0,null,null,null,null,ES,9006962964,null,null,null,null,(1)6446500,0,(1)6446500,null,null,null,null,CI00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,X,null,null,null,null,null,null,1,PJ,N,null,null,null,null,null,null,0,null,Z1,null,null,null,85-MX-Otros,null,null,null,00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,null,null,null,0,null,null,

In [0]:
%sql
--UPDATE dev_arqanalitica.gobiernodato.regla_negocio_great_expectation
--SET parametros = '{"condicion":"FITYP = ''PN''","columna":"NAME4","patron":"^[^,]+(,[^,]+)?$"}'
--WHERE Pkregla = 212;

SELECT *
FROM dev_arqanalitica.gobiernodato.regla_negocio_great_expectation;

Pkregla,dominio,Tabla,Campo,Dimension,NombreRegla,descripcion,NombreReglaGreatExpectation,parametros,fecha,responsable,estado,indicadorTabla,path
66,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,name4,Validez,TAXONOMIA APELLIDO PN,"Si el campo FITYP = ""PJ"" el campo NAME4 debe estar nulo o vacio",expectativa_valores_nulos_con_condicion,"{""condicion"":""fityp = 'PJ'"",""columna"":""name4""}",2025-11-19T07:43:53.94791Z,cmpastrana@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
51,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,uwaer,Integridad,Relación moneda y cliente ext.,"si el campo KTOKD (grupo de cuentas) es igual a """"ZEXT"""" entonces el valor del campo WAERS(moneda) debe ser diferente de """"COP"""" para que “CUMPLA”",expectativa_valores_no_permitidos,"{""condicion"":""ktokd = 'ZEXT'"",""columna"":""uwaer"",""valores"":[""COP""]}",2025-11-19T07:43:58.305924Z,cmpastrana@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
56,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,name3,Validez,STD teléfono móvil,"Si el campo FITYP = ""PN"" el campo NAME3 debe contener solo nombres separados por coma y en mayúscula, por ejemplo: ELIANA,MARCELA",expectativa_con_condicional_y_patron,"{""condicion"":""fityp = 'PN'"",""columna"":""name3"",""patron"":""^[A-Z]+(,[A-Z]+)*$""}",2025-11-11T03:52:49.843065Z,mtrubio@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
61,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,name3,Validez,STD teléfono móvil,"Si el campo FITYP = ""PJ"" el campo NAME3 debe estar nulo o vacio",expectativa_valores_nulos_con_condicion,"{""condicion"":""fityp = 'PJ'"",""columna"":""name3""}",2025-11-19T07:43:52.934193Z,cmpastrana@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
71,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,adrnr,Completitud,STD dirección,El campo no puede ser nulo,expectativa_valores_no_nulos,"{""columna"":""adrnr""}",2025-11-10T21:18:10.466849Z,mtrubio@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
76,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,adrnr,Validez,STD dirección,El campo ADRNR no debe iniciar con espacios ni contener caracteres especiales ni tener menos de 8 caracteres,expectativa_valores_con_patron,"{""columna"" : ""adrnr"", ""patron"": ""^(?!s)[A-Za-z0-9]{8,}$""}",2025-11-19T07:43:44.090749Z,cmpastrana@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
81,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,adrnr,Validez,STD dirección,"Si el campo LAND1(pais) es igual a ""CO"" Extraer del campo ADRNR la primera palabra hasta encontrar el primer espacio y si existe en la lista de referencia LR006 ""CUMPLE"" de lo contrario ""NO CUMPLE""",expectativa_primera_palabra_en_referencia_con_condicion,"{""condicion"":""land1 = 'CO'"",""columna"":""adrnr"",""dominio"":""Clientes"",""referencia"":""LR006""}",2025-11-19T07:43:45.1985Z,cmpastrana@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
86,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,ernam,Integridad,Seguridad en creación PSA/VTC,"Si el campo BEGRU(autorización) es igual a ""PSA"" o ""VTC"" el valor del campo ERNAM debe existir en la LR001",expectativa_condicional_tabla_referencia,"{""condicion"": ""begru = 'PSA' OR begru = 'VTC'"", ""columna"": ""ernam"", ""dominio"": ""Clientes"", ""codigo"": ""LR001""}",2025-11-19T07:43:40.500538Z,cmpastrana@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
91,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,ernam,Validez,Seguridad en creación USC,El campo debe corresponder a la lista de referencia definida LR001,expectativa_valores_por_referencia,"{""columna"":""ernam"",""dominio"":""Clientes"",""codigo"":""LR001""}",202

In [0]:
# 2) Regla GE desde tabla de gobierno

rule = load_rule_from_table(RULES_TABLE, PKREGLA)
expectation, params_rule, tipo_rule = build_expectation_from_governance_rule(rule)

print(
    "Regla cargada:",
    {
        "Pkregla": rule.get("Pkregla"),
        "tipo_expectativa": rule.get("NombreReglaGreatExpectation"),
    },
)


Regla cargada: {'Pkregla': 218, 'tipo_expectativa': 'expectativa_condicional_valores_en_lista'}


/databricks/spark/python/pyspark/sql/pandas/utils.py:85: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pyarrow.__version__) < LooseVersion(minimum_pyarrow_version):


In [0]:
# 3) Validacion GE
context = get_test_context()
datasource = get_or_create_datasource(context)
asset_name = f"asset_sandbox_{datetime.now().strftime('%H%M%S')}"
data_asset = get_or_create_asset(datasource, asset_name)
batch_definition = get_or_create_batch_definition(data_asset)

suite_name = f"suite_sandbox_{datetime.now().strftime('%H%M%S')}"
suite = create_or_update_suite(context, suite_name, [expectation])
validation_def = create_validation_definition(context, batch_definition, suite, suite_name)
checkpoint = create_or_update_checkpoint(
    context,
    f"checkpoint_{suite_name}",
    [validation_def],
)

run_id = generate_run_identifier(prefix="run_sandbox")
result = checkpoint.run(batch_parameters={"dataframe": df}, run_id=run_id)


/databricks/python/lib/python3.12/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)


Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

{"ts": "2026-04-21 21:05:20.443", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `BUKRS` cannot be resolved. Did you mean one of the following? [`BUBKZ`, `UF`, `ALC`, `BAHNS`, `BBBNR`]. SQLSTATE: 42703; line 1 pos 0;\n'Aggregate [unresolvedalias(count(1))]\n+- 'Filter ('BUKRS = DIC0)\n   +- Project [_0#41001L AS LIFNR#41279L, _1#41002 AS LAND1#41280, _2#41003 AS NAME1#41281, _3#41004 AS NAME2#41282, _4#41005 AS NAME3#41283, _5#41006 AS NAME4#41284, _6#41007 AS ORT01#41285, _7#41008 AS ORT02#41286, _8#41009 AS PFACH#41287, _9#41010 AS PSTL2#41288, _10#41011L AS PSTLZ#41289L, _11#41012 AS REGIO#41290, _12#41013 AS SORTL#41291, _13#41014 AS STRAS#41292, _14#41015L AS ADRNR#41293L, _15#41016 AS MCOD1#41294, _16#41017 AS MCOD2#41295, _17#41018 AS MCOD3#41296, _18#41019 AS ANRED#41297, _19#41020 AS BAHNS#41298, _20#41021L AS BBBNR#41299L, _21#41022L AS BBSNR#41300L, _22#41023 AS BEGRU#41301,

In [0]:
run_result = list(result.run_results.values())[0]

# Compatibilidad entre estructuras de salida GE
if hasattr(run_result, "to_json_dict"):
    validation_json = run_result.to_json_dict()
elif hasattr(run_result, "validation_result") and hasattr(run_result.validation_result, "to_json_dict"):
    validation_json = run_result.validation_result.to_json_dict()
else:
    raise ValueError(f"Estructura de run_result no soportada: {type(run_result)}")

stats = validation_json.get("statistics", {})

summary = {
    "success": validation_json.get("success"),
    "evaluated_expectations": stats.get("evaluated_expectations"),
    "successful_expectations": stats.get("successful_expectations"),
    "unsuccessful_expectations": stats.get("unsuccessful_expectations"),
    "success_percent": stats.get("success_percent"),
}
print("Resumen GE:", summary)
display(spark.createDataFrame([summary]))

# Detalle GE por estado: cumplen, no_aplican y fallan
passed = []
not_applicable = []
failed = []

total_rows = df.count()

for idx, r in enumerate(validation_json.get("results", []), start=1):
    cfg = r.get("expectation_config", {}) or {}
    kwargs = cfg.get("kwargs", {}) or {}
    result_info = r.get("result", {}) or {}
    row_condition = kwargs.get("row_condition")

    if result_info.get("element_count") is not None:
        aplica = int(result_info.get("element_count"))
    elif row_condition:
        try:
            aplica = df.where(expr(row_condition)).count()
        except Exception:
            aplica = total_rows
    else:
        aplica = total_rows

    payload = {
        "indice": idx,
        "expectation_type": cfg.get("type"),
        "success": bool(r.get("success")),
        "aplica_registros": aplica,
        "kwargs": kwargs,
        "result": result_info,
    }

    if aplica == 0:
        not_applicable.append(payload)
    elif r.get("success"):
        passed.append(payload)
    else:
        failed.append(payload)

status_summary = {
    "passed_count": len(passed),
    "not_applicable_count": len(not_applicable),
    "failed_count": len(failed),
    "passed_expectations": passed,
    "not_applicable_expectations": not_applicable,
    "failed_expectations": failed,
}

print("Estado GE (JSON):")
print(json.dumps(status_summary, ensure_ascii=False, indent=2, default=str))

if failed:
    failed_rows = []
    for f in failed:
        failed_rows.append({
            "indice": f["indice"],
            "expectation_type": f.get("expectation_type"),
            "kwargs_json": json.dumps(f.get("kwargs", {}), ensure_ascii=False),
            "result_json": json.dumps(f.get("result", {}), ensure_ascii=False),
        })
    display(spark.createDataFrame(failed_rows))



Resumen GE: {'success': False, 'evaluated_expectations': 1, 'successful_expectations': 0, 'unsuccessful_expectations': 1, 'success_percent': 0.0}


/databricks/spark/python/pyspark/sql/pandas/utils.py:85: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pyarrow.__version__) < LooseVersion(minimum_pyarrow_version):


evaluated_expectations,success,success_percent,successful_expectations,unsuccessful_expectations
1,false,0.0,0,1


{"ts": "2026-04-21 21:05:22.004", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `BUKRS` cannot be resolved. Did you mean one of the following? [`BUBKZ`, `UF`, `ALC`, `BAHNS`, `BBBNR`]. SQLSTATE: 42703; line 1 pos 0;\n'Aggregate [unresolvedalias(count(1))]\n+- 'Filter ('BUKRS = DIC0)\n   +- Project [_0#41001L AS LIFNR#41279L, _1#41002 AS LAND1#41280, _2#41003 AS NAME1#41281, _3#41004 AS NAME2#41282, _4#41005 AS NAME3#41283, _5#41006 AS NAME4#41284, _6#41007 AS ORT01#41285, _7#41008 AS ORT02#41286, _8#41009 AS PFACH#41287, _9#41010 AS PSTL2#41288, _10#41011L AS PSTLZ#41289L, _11#41012 AS REGIO#41290, _12#41013 AS SORTL#41291, _13#41014 AS STRAS#41292, _14#41015L AS ADRNR#41293L, _15#41016 AS MCOD1#41294, _16#41017 AS MCOD2#41295, _17#41018 AS MCOD3#41296, _18#41019 AS ANRED#41297, _19#41020 AS BAHNS#41298, _20#41021L AS BBBNR#41299L, _21#41022L AS BBSNR#41300L, _22#41023 AS BEGRU#41301,

Estado GE (JSON):
{
  "passed_count": 0,
  "not_applicable_count": 0,
  "failed_count": 1,
  "passed_expectations": [],
  "not_applicable_expectations": [],
  "failed_expectations": [
    {
      "indice": 1,
      "expectation_type": "expect_column_values_to_be_in_set",
      "success": false,
      "aplica_registros": 99,
      "kwargs": {
        "column": "BLNKZ",
        "row_condition": "BUKRS = 'DIC0'",
        "condition_parser": "spark",
        "value_set": [
          "01"
        ],
        "batch_id": "spark_datasource_sandbox-asset_sandbox_210519"
      },
      "result": {}
    }
  ]
}


expectation_type,indice,kwargs_json,result_json
expect_column_values_to_be_in_set,1,"{""column"": ""BLNKZ"", ""row_condition"": ""BUKRS = 'DIC0'"", ""condition_parser"": ""spark"", ""value_set"": [""01""], ""batch_id"": ""spark_datasource_sandbox-asset_sandbox_210519""}",{}


In [0]:
# 4) Resumen negocio generico (sin hardcode por tipo)

def quality_summary_from_ge(df_in, validation_json):
    total = df_in.count()
    results = validation_json.get("results", [])

    if not results:
        return spark.createDataFrame(
            [(total, 0, 0, total, 0.0)],
            ["total", "cumple", "no_cumple", "no_aplica", "pct_calidad"],
        )

    # En este notebook validamos 1 expectativa por corrida
    r0 = results[0]
    cfg = r0.get("expectation_config", {})
    kwargs = cfg.get("kwargs", {}) or {}
    r = r0.get("result", {}) or {}

    row_condition = kwargs.get("row_condition")

    # 1) filas aplicables
    if r.get("element_count") is not None:
        aplica = int(r.get("element_count"))
    elif row_condition:
        try:
            aplica = df_in.where(expr(row_condition)).count()
        except Exception:
            aplica = total
    else:
        aplica = total

    # 2) filas no cumple
    unexpected_count = int(r.get("unexpected_count") or 0)
    missing_count = int(r.get("missing_count") or 0)
    # En reglas de lista condicional, missing tambien se considera NO CUMPLE
    no_cumple = min(unexpected_count + missing_count, aplica)

    cumple = max(aplica - no_cumple, 0)
    no_aplica = max(total - aplica, 0)
    pct = round((cumple / aplica) * 100, 1) if aplica > 0 else 0.0

    return spark.createDataFrame(
        [(total, cumple, no_cumple, no_aplica, pct)],
        ["total", "cumple", "no_cumple", "no_aplica", "pct_calidad"],
    )

resumen_calidad = quality_summary_from_ge(df, validation_json)
display(resumen_calidad)


{"ts": "2026-04-21 21:05:23.006", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `BUKRS` cannot be resolved. Did you mean one of the following? [`BUBKZ`, `UF`, `ALC`, `BAHNS`, `BBBNR`]. SQLSTATE: 42703; line 1 pos 0;\n'Aggregate [unresolvedalias(count(1))]\n+- 'Filter ('BUKRS = DIC0)\n   +- Project [_0#41001L AS LIFNR#41279L, _1#41002 AS LAND1#41280, _2#41003 AS NAME1#41281, _3#41004 AS NAME2#41282, _4#41005 AS NAME3#41283, _5#41006 AS NAME4#41284, _6#41007 AS ORT01#41285, _7#41008 AS ORT02#41286, _8#41009 AS PFACH#41287, _9#41010 AS PSTL2#41288, _10#41011L AS PSTLZ#41289L, _11#41012 AS REGIO#41290, _12#41013 AS SORTL#41291, _13#41014 AS STRAS#41292, _14#41015L AS ADRNR#41293L, _15#41016 AS MCOD1#41294, _16#41017 AS MCOD2#41295, _17#41018 AS MCOD3#41296, _18#41019 AS ANRED#41297, _19#41020 AS BAHNS#41298, _20#41021L AS BBBNR#41299L, _21#41022L AS BBSNR#41300L, _22#41023 AS BEGRU#41301,

total,cumple,no_cumple,no_aplica,pct_calidad
99,0,99,0,0.0
